# 8.7. Densely Connected Networks (DenseNet)
D2L의 Densely Connected Networks (DenseNet)장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 1. DenseNet이란?

ResNet은 이전 입력을 다음 층의 출력과 더해서 정보를 전달한다.

$$
Y = F(X) + X
$$

DenseNet은 조금 다르다. 이전 특징을 더하지 않고 그대로 이어 붙인다.

$$
Y = [X, F(X)]
$$

[ ]는 덧셈이 아니라 채널 방향 concatenation을 의미한다.

예를 들어서

기존 feature map: 64채널
새로운 feature map: 32채널

ResNet -> 64채널 + 64채널 = 64채널
DenseNet -> 64채널과 32채널 연결 = 96채널

DenseNet의 핵심 아이디어는 이전에 추출한 특징을 버리지 않고 계속 다음 층에서 재사용하는 것이다.

## 2. ResNet과 DenseNet의 차이

```text
ResNet:
X ─────────────┐
│              +
└→ Conv → F(X) ┘
      ↓
   X + F(X)

DenseNet:
X ───────────────┐
│                │
└→ Conv → F(X)   │
                 ↓
             [X, F(X)]
```

ResNet은 특징을 더하고, DenseNet은 특징을 쌓는다. DenseNet에서는 깊어질수록 다음과 같이 입력되는 정보가 많아진다.

```text
Layer 1: X
Layer 2: [X, F1]
Layer 3: [X, F1, F2]
Layer 4: [X, F1, F2, F3]
```

뒤쪽 레이어는 앞쪽에서 만들어진 거의 모든 특징을 직접 사용할 수 있다.

## 3. DenseNet의 전체 구조

DenseNet은 크게 두 종류의 블록으로 구성된다.

Dense Block -> 특징을 계속 생성하고 연결한다.

Transition Layer -> 너무 많아진 채널과 H, W를 줄인다.

전체 구조는 이렇다.
```text
입력
 ↓
Conv + Pool
 ↓
Dense Block
 ↓
Transition
 ↓
Dense Block
 ↓
Transition
 ↓
Dense Block
 ↓
Transition
 ↓
Dense Block
 ↓
Global Average Pooling
 ↓
Linear
 ↓
분류 결과
```

Dense Block에서는 채널이 증가하고 Transition Layer에서는 다시 압축한다.

## 4. DenseNet의 Conv Block

DenseNet의 기본 Conv Block은 다음 구조를 사용한다.

BatchNorm -> ReLU -> 3×3 Conv

PyTorch로 하면
```python
import torch
from torch import nn

def conv_block(out_channels):
    return nn.Sequential(
        nn.LazyBatchNorm2d(),
        nn.ReLU(),
        nn.LazyConv2d(
            out_channels,
            kernel_size=3,
            padding=1
        )
    )
```

padding=1이므로 3×3 convolution을 사용해도 H와 W는 유지된다.

중요한 점은 Conv Block 자체가 입력을 연결하는 것이 아니라, Dense Block에서 Conv 결과와 기존 입력을 연결한다는 것이다.

## 5. Dense Block

Dense Block에는 여러 개의 Conv Block이 들어간다.

```python
class DenseBlock(nn.Module):
    def __init__(self, num_convs, growth_rate):
        super().__init__()

        self.blocks = nn.ModuleList([
            conv_block(growth_rate)
            for _ in range(num_convs)
        ])

    def forward(self, X):
        for block in self.blocks:
            Y = block(X)
            X = torch.cat((X, Y), dim=1) # [N, C, H, W] 에서 dim=1이기 때문에 채널 방향이다

        return X
```
기존 특징과 새롭게 추출한 특징을 채널 방향으로 계속 붙인다.

## 6. Growth Rate

DenseNet에서는 Conv Block 하나가 몇 개의 채널을 추가할지를 growth rate라고 부른다.
```python
block = DenseBlock(
    num_convs=2,
    growth_rate=10
)

X = torch.randn(4, 3, 8, 8)
Y = block(X)

print(Y.shape)
```

입력 3채널에서 

첫 번째 Conv
3 + 10 = 13채널

두 번째 Conv
13 + 10 = 23채널

[4, 3, 8, 8] -> [4, 23, 8, 8]

일반적으로 Dense Block의 출력 채널은:

$$
C_{out}=C_{in}+L\times k
$$

$C_{in}$: 입력 채널
$L$: Conv Block 개수
$k$: growth rate

## 7. 왜 Transition Layer가 필요한가?

Dense Block을 계속 통과하면 채널 수가 계속 증가한다.

예를 들어 growth rate가 32라면 64 -> 96 -> 128 -> 160 -> 192 -> ...

채널이 끝없이 증가하면 계산량과 GPU 메모리 사용량이 너무 커진다.

그래서 Dense Block 사이에 Transition Layer를 넣어서 크기를 줄인다.

구조는 이렇다.

```text
BatchNorm
↓
ReLU
↓
1×1 Conv
↓
2×2 Average Pooling, stride=2
```

```python
def transition_block(out_channels):
    return nn.Sequential(
        nn.LazyBatchNorm2d(),
        nn.ReLU(),
        nn.LazyConv2d(
            out_channels,
            kernel_size=1
        ),
        nn.AvgPool2d(
            kernel_size=2,
            stride=2
        )
    )
```
1×1 Conv -> 채널 C 감소  
Average Pooling -> H, W 감소

## 8. Transition Layer의 Shape 변화

예를 들어 Dense Block의 출력이 [4, 23, 8, 8]이라고 했을때

[4, 23, 8, 8] -> 1×1 Conv -> [4, 10, 8, 8] -> AvgPool 2×2, stride=2 -> [4, 10, 4, 4]

Dense Block은 특징을 축적하고, Transition Layer는 ​축적된 특징을 압축한다고 이해하면 된다.

## 9. DenseNet 모델 구조

DenseNet의 첫 부분은 ResNet과 비슷하다.

```python
stem = nn.Sequential(
    nn.LazyConv2d(
        64,
        kernel_size=7,
        stride=2,
        padding=3
    ),
    nn.LazyBatchNorm2d(),
    nn.ReLU(),
    nn.MaxPool2d(
        kernel_size=3,
        stride=2,
        padding=1
    )
)
```

이후 여러 Dense Block을 사용한다. 예를 들어서

growth_rate = 32이고 Dense Block 하나당 Conv = 4개 라면 하나의 Dense Block이 추가하는 채널 수는 4 x 32 = 128개이다.

전체적인 흐름이다.

```text
Conv 7×7
↓
MaxPool
↓
Dense Block × 4
↓
Transition
↓
Dense Block × 4
↓
Transition
↓
Dense Block × 4
↓
Transition
↓
Dense Block × 4
↓
BatchNorm + ReLU
↓
Global Average Pooling
↓
Linear
```

마지막에는 모든 공간 정보를 Global Average Pooling으로 하나씩 압축하고 Linear Layer를 이용해 클래스를 예측한다.

## 10. DenseNet의 핵심을 Shape으로 이해하기

예를 들어 입력이 [N, 64, 24, 24]이고 growth rate가 32인 Dense Block에 Conv Block 4개가 있다면

    64 -> 96 -> 128 -> 160 -> 192

Dense Block 출력은 이렇다. [N, 192, 24, 24]

Transition Layer에서 채널을 절반으로 줄인다면

    [N, 192, 24, 24] -> 1×1 Conv -> [N, 96, 24, 24] -> AvgPool -> [N, 96, 12, 12]

DenseNet은 반복적으로 이런 구조이다.

    특징 생성 -> 특징 누적 -> 압축 -> 특징 생성 -> 특징 누적 -> 압축

## 11. DenseNet의 장점과 단점
### 장점

DenseNet의 가장 큰 특징은 feature reuse다. 초기 레이어가 만든 단순한 특징을 뒤쪽 레이어가 직접 다시 사용할 수 있다.

초기 layer -> edge, texture 같은 특징  
중간 layer -> 부분적인 모양  
깊은 layer -> 복잡한 객체 특징  

뒤쪽 레이어는 앞에서 만든 특징을 다시 처음부터 만들 필요 없이 그대로 활용할 수 있다. 그리고 gradient가 앞쪽 레이어까지 전달될 수 있는 경로가 많아 깊은 네트워크의 학습에도 유리하다.

### 단점

메모리 사용량이 문제이다.

```text
[X]
[X, F1]
[X, F1, F2]
[X, F1, F2, F3]
...
```
feature map을 계속 가지고 있어야 하기 때문에 GPU 메모리를 많이 소비할 수 있다. 

특징 재사용, gradient 전달 좋아지지만 메모리 사용량은 올라가는 trade-off가 존재한다.

## 12. ResNet vs DenseNet

| 구분     | ResNet         | DenseNet         |
| ------ | -------------- | ---------------- |
| 연결 방식  | Addition       | Concatenation    |
| 표현     | $X + F(X)$     | $[X, F(X)]$      |
| 채널 수   | 보통 유지          | 계속 증가            |
| 이전 특징  | 합쳐짐            | 그대로 보존           |
| 채널 제어  | Residual Block | Transition Layer |
| 특징 재사용 | 가능             | 매우 적극적           |
| 메모리 사용 | 상대적으로 적음       | 상대적으로 많음         |


ResNet -> 기존 특징 + 새로운 특징  
DenseNet -> 기존 특징 | 새로운 특징

DenseNet에서는 기존 feature map을 없애지 않고 새로운 feature map을 옆에 계속 붙여 나간다.

## 13. 오늘의 정리

- DenseNet은 ResNet의 아이디어를 확장한 CNN 구조다.
- ResNet은 feature map을 더하지만 DenseNet은 채널 방향으로 연결한다.
- torch.cat(..., dim=1)이 DenseNet의 핵심 연산이다.
- Dense Block에서는 이전 모든 특징을 계속 재사용한다.
- Conv Block 하나가 추가하는 채널 수를 growth rate라고 한다.
- Dense Block을 지날수록 채널 수가 계속 증가한다.
- Transition Layer의 1×1 Conv가 채널 수를 줄인다.
- Average Pooling(stride=2)은 H와 W를 절반으로 줄인다.
- 전체 흐름은 Dense Block → 압축 → Dense Block → 압축의 반복이다.
- DenseNet은 feature reuse가 강력하지만 feature map을 많이 저장하므로 GPU 메모리 사용량이 커질 수 있다.
- 핵심 비교는 ResNet = addition, DenseNet = concatenation이다.